# Lab 1.4: Validation, Retry, and Feedback Loops

Your extraction pipeline from Lab 1.3 is live. It produces structured output reliably -- but the extracted **values** sometimes have errors. Date formats wrong, totals that do not add up, fields that the model misread.

**What you will learn:**
- Why blind retry (same prompt, no feedback) wastes API calls and does not fix errors
- Why retrying for absent information makes the model fabricate harder
- How to build a validation function that catches format, math, and consistency errors
- How to decide whether a retry would help (format error) or hurt (absent data)
- How to build retry prompts with specific error feedback
- How to wire it all together into an extract-validate-retry loop

**The pattern:** Validate extraction output. If errors are fixable (format, math), retry with specific error feedback. If information is absent, accept null and move on.

## Setup

In [ ]:
import anthropic
import json
import re
from datetime import datetime

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

## Test Data

Three documents with known extraction issues, plus their (intentionally flawed) extractions.

In [ ]:
# --- Document 1: Format errors (dates in wrong format) ---

DOCUMENT_FORMAT_ERRORS = """
INVOICE #INV-2024-0331
Vendor: Precision Parts Ltd.
Date: March 15, 2024

Items:
  1. Titanium Rod TR-100 -- Qty: 5 -- Unit: $120.00 -- Total: $600.00
  2. Steel Plate SP-50 -- Qty: 10 -- Unit: $45.00 -- Total: $450.00

Subtotal: $1,050.00
Tax (9%): $94.50
Total: $1,144.50

Payment: Paid via wire transfer on March 20, 2024
"""

EXTRACTION_FORMAT_ERRORS = {
    "invoice_number": "INV-2024-0331",
    "vendor_name": "Precision Parts Ltd.",
    "purchase_date": "03/15/2024",        # WRONG: should be 2024-03-15
    "line_items": [
        {"description": "Titanium Rod TR-100", "quantity": 5, "unit_price": 120.00, "line_total": 600.00},
        {"description": "Steel Plate SP-50", "quantity": 10, "unit_price": 45.00, "line_total": 450.00},
    ],
    "subtotal": 1050.00,
    "tax": 94.50,
    "shipping": None,
    "total": 1144.50,
    "payment_status": "paid",
    "payment_date": "03/20/2024",           # WRONG: should be 2024-03-20
    "warranty_expiry": None,
}

# --- Document 2: Math error (total does not match) ---

DOCUMENT_MATH_ERROR = """
INVOICE #INV-2024-0445
Vendor: QuickShip Supplies
Date: 2024-06-01

Items:
  1. Bubble Wrap Roll (Large) -- Qty: 20 -- Unit: $12.00 -- Total: $240.00
  2. Packing Tape 3-inch -- Qty: 50 -- Unit: $3.50 -- Total: $175.00
  3. Shipping Labels (Roll of 500) -- Qty: 4 -- Unit: $18.00 -- Total: $72.00

Subtotal: $487.00
Tax (8%): $38.96
Shipping: $15.00
Total: $540.96

Payment: Unpaid -- Due by 2024-07-01
"""

EXTRACTION_MATH_ERROR = {
    "invoice_number": "INV-2024-0445",
    "vendor_name": "QuickShip Supplies",
    "purchase_date": "2024-06-01",
    "line_items": [
        {"description": "Bubble Wrap Roll (Large)", "quantity": 20, "unit_price": 12.00, "line_total": 240.00},
        {"description": "Packing Tape 3-inch", "quantity": 50, "unit_price": 3.50, "line_total": 175.00},
        {"description": "Shipping Labels (Roll of 500)", "quantity": 4, "unit_price": 18.00, "line_total": 72.00},
    ],
    "subtotal": 487.00,
    "tax": 38.96,
    "shipping": 15.00,
    "total": 580.96,                         # WRONG: should be 540.96
    "payment_status": "unpaid",
    "warranty_expiry": None,
}

# --- Document 3: Missing warranty (correctly absent) ---

DOCUMENT_MISSING_WARRANTY = """
INVOICE #INV-2024-0512
Vendor: Office Basics Inc.
Date: 2024-07-10

Items:
  1. Copy Paper A4 -- Qty: 100 -- Unit: $5.00 -- Total: $500.00

Total: $500.00
Payment: Paid
"""

EXTRACTION_MISSING_WARRANTY = {
    "invoice_number": "INV-2024-0512",
    "vendor_name": "Office Basics Inc.",
    "purchase_date": "2024-07-10",
    "line_items": [
        {"description": "Copy Paper A4", "quantity": 100, "unit_price": 5.00, "line_total": 500.00},
    ],
    "subtotal": 500.00,
    "tax": None,
    "shipping": None,
    "total": 500.00,
    "payment_status": "paid",
    "warranty_expiry": None,
}

print("Test data loaded:")
print("  1. Format errors: dates in MM/DD/YYYY instead of YYYY-MM-DD")
print("  2. Math error: total is $580.96 but should be $540.96")
print("  3. Missing warranty: correctly null (no errors)")

In [ ]:
# Validation schema reference
EXTRACTION_SCHEMA = {
    "invoice_number": {"type": "string", "required": True},
    "vendor_name": {"type": "string", "required": True},
    "purchase_date": {"type": "string", "format": "date-iso8601", "required": True},
    "line_items": {"type": "array", "required": True},
    "subtotal": {"type": "number", "nullable": True},
    "tax": {"type": "number", "nullable": True},
    "shipping": {"type": "number", "nullable": True},
    "total": {"type": "number", "required": True},
    "payment_status": {
        "type": "string",
        "enum": ["paid", "unpaid", "partial", "unclear"],
        "required": True,
    },
    "payment_date": {"type": "string", "format": "date-iso8601", "nullable": True},
    "warranty_expiry": {"type": "string", "format": "date-iso8601", "nullable": True},
}

print("Validation schema loaded.")
print("  Date fields must be ISO 8601 (YYYY-MM-DD)")
print("  Numeric consistency: subtotal + tax + shipping should equal total")
print("  Nullable fields: null is valid, fabricated values are not")

---
## Phase 1: THE WRONG WAY

Three common retry mistakes that waste money, do not fix errors, and can make things worse.

### Wrong Way #1: Blind retry -- same prompt, no feedback

The most common first instinct: "It got it wrong, just try again."

In [ ]:
# Extraction tool from Lab 1.3
extraction_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document. Use null for absent fields.",
    "input_schema": {
        "type": "object",
        "required": ["invoice_number", "vendor_name", "purchase_date", "line_items", "total", "payment_status"],
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "purchase_date": {"type": "string", "description": "Date in ISO 8601 format: YYYY-MM-DD"},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit_price": {"type": "number"},
                        "line_total": {"type": "number"}
                    }
                }
            },
            "subtotal": {"type": ["number", "null"]},
            "tax": {"type": ["number", "null"]},
            "shipping": {"type": ["number", "null"]},
            "total": {"type": "number"},
            "payment_status": {"type": "string", "enum": ["paid", "unpaid", "partial", "unclear"]},
            "payment_date": {"type": ["string", "null"], "description": "Date in ISO 8601 format: YYYY-MM-DD"},
            "warranty_expiry": {"type": ["string", "null"], "description": "Date in ISO 8601 format: YYYY-MM-DD. Null if no warranty info."},
        }
    }
}


def extract_once(document):
    """Single extraction call using tool_use."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        tools=[extraction_tool],
        tool_choice={"type": "tool", "name": "extract_invoice"},
        messages=[{"role": "user", "content": f"Extract invoice data:\n{document}"}]
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return {}


def quick_validate(extraction):
    """Quick check for the known error types."""
    errors = []
    # Check date format
    for field in ["purchase_date", "payment_date", "warranty_expiry"]:
        val = extraction.get(field)
        if val is not None:
            try:
                datetime.strptime(val, "%Y-%m-%d")
            except ValueError:
                errors.append(f"{field} '{val}' is not ISO 8601 format (YYYY-MM-DD)")
    # Check total consistency
    subtotal = extraction.get("subtotal")
    tax = extraction.get("tax")
    shipping = extraction.get("shipping")
    total = extraction.get("total")
    if all(v is not None for v in [subtotal, tax, total]):
        expected = subtotal + tax + (shipping or 0)
        if abs(expected - total) > 0.02:
            errors.append(f"Total mismatch: subtotal({subtotal}) + tax({tax}) + shipping({shipping or 0}) = {expected}, but total is {total}")
    return errors


print("Extraction and validation functions ready.")

In [ ]:
print("=" * 60)
print("BLIND RETRY: Same prompt, no feedback")
print("=" * 60)

# Simulate blind retry on the math error document
# We use our extraction tool which has the date format in the description,
# so dates will likely be correct. Instead, let us simulate the blind retry
# pattern with a document that produces inconsistent results.

print("\n--- Math Error Document: 3 blind retries ---")
print("(The document says $540.96 total, but extraction might still miscompute)\n")

for attempt in range(3):
    result = extract_once(DOCUMENT_MATH_ERROR)
    errors = quick_validate(result)
    total = result.get("total")
    print(f"  Attempt {attempt + 1}: total=${total}  errors={errors if errors else 'none'}")

print("\nWith tool_use, the model reads the document correctly most of the time.")
print("But when errors DO occur, blind retry has no way to guide correction.")
print("The model has no idea what went wrong on the previous attempt.")

### Wrong Way #2: Retrying for absent information -- fabrication escalation

This is the most dangerous mistake. When information is genuinely absent from the document, retrying pushes the model to try harder to find something -- which means it **fabricates** more confidently on each retry.

In [ ]:
print("=" * 60)
print("RETRYING FOR ABSENT INFORMATION: Fabrication escalation")
print("=" * 60)

# Use a tool that requires warranty_expiry as a non-nullable string
fabrication_tool = {
    "name": "extract_invoice",
    "description": "Extract structured data from an invoice document.",
    "input_schema": {
        "type": "object",
        "required": ["invoice_number", "vendor_name", "total", "warranty_expiry"],
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor_name": {"type": "string"},
            "total": {"type": "number"},
            "warranty_expiry": {"type": "string", "description": "Warranty expiration date in YYYY-MM-DD format"},
        }
    }
}

# Simulate escalating retry prompts for missing warranty
retry_prompts = [
    # Attempt 1: basic extraction
    f"Extract invoice data:\n{DOCUMENT_MISSING_WARRANTY}",
    # Attempt 2: "you missed warranty, try again"
    f"""The previous extraction was missing the warranty_expiry field.
Please re-extract and make sure to include the warranty expiration date.

Document:\n{DOCUMENT_MISSING_WARRANTY}""",
    # Attempt 3: "try harder"
    f"""The warranty_expiry field is still missing. Look carefully at the document
for any mention of warranty, guarantee, or coverage period. Extract the date.

Document:\n{DOCUMENT_MISSING_WARRANTY}""",
]

print("\n--- Missing Warranty Document: escalating retry pressure ---")
print("(There is NO warranty info in this document at all)\n")

for i, prompt in enumerate(retry_prompts):
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        tools=[fabrication_tool],
        tool_choice={"type": "tool", "name": "extract_invoice"},
        messages=[{"role": "user", "content": prompt}]
    )
    for block in response.content:
        if block.type == "tool_use":
            warranty = block.input.get("warranty_expiry", "MISSING")
            print(f"  Attempt {i + 1}: warranty_expiry = {warranty!r}")
            break

print("\nNotice how the model invents warranty dates with increasing confidence.")
print("Each retry that says 'you missed it, try harder' teaches the model that")
print("returning null is wrong -- so it fabricates. This is why you MUST NOT")
print("retry when information is genuinely absent.")

### Wrong Way #3: No cost accounting

Even when retries could help, blindly retrying without tracking cost means you might spend 3x on a document that had a trivial formatting issue.

In [ ]:
print("=" * 60)
print("COST OF BLIND RETRY")
print("=" * 60)

# Approximate token costs
APPROX_INPUT_TOKENS = 500   # per document extraction
APPROX_OUTPUT_TOKENS = 300  # per extraction response
INPUT_COST_PER_1K = 0.003   # Claude Sonnet approximate
OUTPUT_COST_PER_1K = 0.015

cost_per_call = (APPROX_INPUT_TOKENS * INPUT_COST_PER_1K / 1000) + (APPROX_OUTPUT_TOKENS * OUTPUT_COST_PER_1K / 1000)

scenarios = {
    "No retry (1 call)": 1,
    "Blind retry (always 3 calls)": 3,
    "Smart retry (1.2 avg calls)": 1.2,
}

docs_per_month = 10000

print(f"\nAssumptions: ~{APPROX_INPUT_TOKENS} input tokens, ~{APPROX_OUTPUT_TOKENS} output tokens per call")
print(f"Processing {docs_per_month:,} documents per month\n")

for name, avg_calls in scenarios.items():
    monthly_cost = docs_per_month * avg_calls * cost_per_call
    print(f"  {name}: ${monthly_cost:,.2f}/month")

print("\nSmart retry: only retry when validation fails AND the error is fixable.")
print("Most documents pass on the first try.")

### Phase 1 Summary

| Mistake | Consequence |
|---|---|
| Blind retry (no feedback) | Same error every time -- the model has no guidance on what went wrong |
| Retry for absent information | Model fabricates with increasing confidence on each retry |
| No cost accounting | 3x cost for marginal improvement; most documents pass on first try |

---
## Phase 2: THE RIGHT WAY

Build a validation-retry loop with three components:
1. **Validate**: catch format errors, math errors, and consistency issues
2. **Decide**: should we retry (fixable error) or accept (absent data)?
3. **Retry with feedback**: tell the model exactly what went wrong

### Step 1: Validation function

Check the extraction output against the schema. Return a list of specific error messages.

In [ ]:
def validate_extraction(extraction, schema):
    """Validate extraction against schema. Return list of error strings."""
    errors = []
    
    # Check required non-nullable fields
    for field, rules in schema.items():
        if rules.get("required") and not rules.get("nullable"):
            if field not in extraction or extraction[field] is None:
                errors.append(f"Required field '{field}' is missing or null")
    
    # Check date formats (ISO 8601: YYYY-MM-DD)
    for field, rules in schema.items():
        if rules.get("format") == "date-iso8601":
            val = extraction.get(field)
            if val is not None:  # null is ok for nullable date fields
                try:
                    datetime.strptime(val, "%Y-%m-%d")
                except (ValueError, TypeError):
                    errors.append(f"{field} '{val}' is not ISO 8601 format (expected YYYY-MM-DD)")
    
    # Check enum values
    for field, rules in schema.items():
        if "enum" in rules:
            val = extraction.get(field)
            if val is not None and val not in rules["enum"]:
                errors.append(f"{field} '{val}' is not a valid value (expected one of {rules['enum']})")
    
    # Check numeric consistency: line_items sum vs subtotal
    line_items = extraction.get("line_items", [])
    subtotal = extraction.get("subtotal")
    if line_items and subtotal is not None:
        line_sum = sum(item.get("line_total", 0) for item in line_items)
        if abs(line_sum - subtotal) > 0.02:
            errors.append(f"Line items sum ({line_sum}) does not match subtotal ({subtotal})")
    
    # Check numeric consistency: subtotal + tax + shipping vs total
    total = extraction.get("total")
    tax = extraction.get("tax")
    shipping = extraction.get("shipping")
    if subtotal is not None and total is not None:
        expected_total = subtotal + (tax or 0) + (shipping or 0)
        if abs(expected_total - total) > 0.02:
            errors.append(
                f"Total mismatch: subtotal({subtotal}) + tax({tax or 0}) + shipping({shipping or 0}) "
                f"= {expected_total}, but stated total is {total}"
            )
    
    return errors


# Test it
print("=== Validation Tests ===")
print("\nFormat errors document:")
errs = validate_extraction(EXTRACTION_FORMAT_ERRORS, EXTRACTION_SCHEMA)
for e in errs:
    print(f"  - {e}")

print("\nMath error document:")
errs = validate_extraction(EXTRACTION_MATH_ERROR, EXTRACTION_SCHEMA)
for e in errs:
    print(f"  - {e}")

print("\nMissing warranty document (should have NO errors):")
errs = validate_extraction(EXTRACTION_MISSING_WARRANTY, EXTRACTION_SCHEMA)
print(f"  Errors: {errs}")
print(f"  {'PASS' if not errs else 'FAIL'}: null warranty is valid (field is nullable)")

### Step 2: Should we retry?

The critical decision: is the error **fixable** (wrong format, wrong math) or **unfixable** (information genuinely absent from the document)?

In [ ]:
def should_retry(errors, source_document):
    """Decide whether retrying would fix the errors.
    
    Returns True for fixable errors (format, math).
    Returns False for absent data or no errors.
    """
    if not errors:
        return False
    
    # Keywords that indicate the information might be absent
    absent_keywords = {
        "warranty": ["warranty", "guarantee", "coverage"],
        "shipping": ["shipping", "ship to", "delivery"],
        "payment_date": ["paid", "payment"],
    }
    
    has_fixable_error = False
    
    for error in errors:
        error_lower = error.lower()
        
        # Format errors are always fixable
        if "format" in error_lower or "iso 8601" in error_lower or "iso8601" in error_lower:
            has_fixable_error = True
            continue
        
        # Math/consistency errors are fixable
        if any(kw in error_lower for kw in ["mismatch", "sum", "total", "match", "consistency"]):
            has_fixable_error = True
            continue
        
        # Enum errors are fixable
        if "valid value" in error_lower or "enum" in error_lower:
            has_fixable_error = True
            continue
        
        # Check if the error is about absent information
        if "missing" in error_lower or "required" in error_lower or "not present" in error_lower:
            # Check if the field's data exists in the source document
            field_found_in_doc = False
            for field_key, search_terms in absent_keywords.items():
                if field_key in error_lower:
                    for term in search_terms:
                        if term.lower() in source_document.lower():
                            field_found_in_doc = True
                            break
                    break
            
            if field_found_in_doc:
                has_fixable_error = True  # Data exists, extraction just missed it
            # If not found in document, this error is NOT fixable
            continue
        
        # Default: treat unknown errors as potentially fixable
        has_fixable_error = True
    
    return has_fixable_error


# Test it
print("=== Retry Decision Tests ===")

# Format errors -> retry
errs = validate_extraction(EXTRACTION_FORMAT_ERRORS, EXTRACTION_SCHEMA)
decision = should_retry(errs, DOCUMENT_FORMAT_ERRORS)
print(f"\nFormat errors: should_retry = {decision}  {'PASS' if decision else 'FAIL'}")

# Math errors -> retry
errs = validate_extraction(EXTRACTION_MATH_ERROR, EXTRACTION_SCHEMA)
decision = should_retry(errs, DOCUMENT_MATH_ERROR)
print(f"Math error:    should_retry = {decision}  {'PASS' if decision else 'FAIL'}")

# No errors -> no retry
decision = should_retry([], DOCUMENT_MISSING_WARRANTY)
print(f"No errors:     should_retry = {decision}  {'PASS' if not decision else 'FAIL'}")

# Absent warranty -> no retry
absent_errors = ["warranty_expiry is required but not present"]
decision = should_retry(absent_errors, DOCUMENT_MISSING_WARRANTY)
print(f"Absent data:   should_retry = {decision}  {'PASS' if not decision else 'FAIL'}")

### Step 3: Retry prompt with specific error feedback

The retry prompt must include: (1) the original document, (2) the failed extraction, (3) each specific error, and (4) a correction instruction.

In [ ]:
def build_retry_prompt(document, failed_extraction, errors):
    """Build a retry prompt with specific error feedback."""
    error_list = "\n".join(f"  {i+1}. {e}" for i, e in enumerate(errors))
    
    return f"""You previously extracted data from this document, but the extraction has errors.

ORIGINAL DOCUMENT:
{document}

YOUR PREVIOUS EXTRACTION (contains errors):
{json.dumps(failed_extraction, indent=2)}

VALIDATION ERRORS FOUND:
{error_list}

Please re-extract the data, correcting the specific errors listed above.
Keep all correct fields unchanged. Only fix the errors."""


# Show what a retry prompt looks like
errs = validate_extraction(EXTRACTION_FORMAT_ERRORS, EXTRACTION_SCHEMA)
retry_prompt = build_retry_prompt(DOCUMENT_FORMAT_ERRORS, EXTRACTION_FORMAT_ERRORS, errs)

print("=== Example Retry Prompt ===")
print(retry_prompt[:500])
print("...")
print(f"\nTotal prompt length: {len(retry_prompt)} characters")
print("\nKey: the model sees EXACTLY what went wrong and can fix just those fields.")

### Step 4: The complete extract-validate-retry loop

In [ ]:
def extract_with_retry(document, tool, max_retries=2, verbose=True):
    """Complete extraction loop with validation and smart retry."""
    
    # Initial extraction
    if verbose:
        print(f"  Attempt 1: extracting...")
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=4096,
        tools=[tool],
        tool_choice={"type": "tool", "name": tool["name"]},
        messages=[{"role": "user", "content": f"Extract invoice data:\n{document}"}]
    )
    
    extraction = {}
    for block in response.content:
        if block.type == "tool_use":
            extraction = block.input
            break
    
    # Validate
    errors = validate_extraction(extraction, EXTRACTION_SCHEMA)
    
    if verbose:
        if errors:
            print(f"    Errors: {errors}")
        else:
            print(f"    Valid on first try!")
    
    # Retry loop
    retries = 0
    while errors and retries < max_retries:
        # Should we retry?
        if not should_retry(errors, document):
            if verbose:
                print(f"    Skipping retry: errors are due to absent data, not fixable.")
            break
        
        retries += 1
        if verbose:
            print(f"  Attempt {retries + 1}: retrying with error feedback...")
        
        # Build retry prompt with feedback
        retry_prompt = build_retry_prompt(document, extraction, errors)
        
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=[tool],
            tool_choice={"type": "tool", "name": tool["name"]},
            messages=[{"role": "user", "content": retry_prompt}]
        )
        
        for block in response.content:
            if block.type == "tool_use":
                extraction = block.input
                break
        
        # Re-validate
        errors = validate_extraction(extraction, EXTRACTION_SCHEMA)
        if verbose:
            if errors:
                print(f"    Remaining errors: {errors}")
            else:
                print(f"    All errors fixed!")
    
    # Attach remaining errors if any
    if errors:
        extraction["_validation_errors"] = errors
    
    extraction["_retries_used"] = retries
    return extraction


print("extract_with_retry function defined.")

### Test the complete loop

In [ ]:
print("=" * 60)
print("COMPLETE EXTRACTION LOOP")
print("=" * 60)

print("\n--- Document with format errors (dates) ---")
result1 = extract_with_retry(DOCUMENT_FORMAT_ERRORS, extraction_tool)
print(f"  Final purchase_date: {result1.get('purchase_date')!r}")
print(f"  Retries used: {result1.get('_retries_used')}")

print("\n--- Document with math error (total) ---")
result2 = extract_with_retry(DOCUMENT_MATH_ERROR, extraction_tool)
print(f"  Final total: {result2.get('total')}")
print(f"  Retries used: {result2.get('_retries_used')}")

print("\n--- Document with missing warranty (should pass on first try) ---")
result3 = extract_with_retry(DOCUMENT_MISSING_WARRANTY, extraction_tool)
print(f"  warranty_expiry: {result3.get('warranty_expiry')!r}")
print(f"  Retries used: {result3.get('_retries_used')}")
print(f"  Validation errors: {result3.get('_validation_errors', 'none')}")

### Phase 2 Comparison: Blind Retry vs Smart Retry

| Aspect | Blind Retry | Smart Retry |
|---|---|---|
| Error feedback | None | Specific validation errors in prompt |
| Absent data | Retries (fabrication risk) | Skips retry, accepts null |
| Cost | Always max retries | Retries only when fixable |
| Fix rate | Random | Targeted |

---
## Phase 3: YOUR TURN

Build a complete validation-retry loop for a set of 5 documents. Each document has a different error pattern. You must:

1. **Validate** each extraction correctly
2. **Decide** whether to retry or accept
3. **Retry with feedback** when appropriate
4. **Track** which error patterns were seen (for upstream feedback)

In [ ]:
# Five documents with different error patterns

CHALLENGE_DOCS = {
    "doc_perfect": {
        "text": """
INVOICE #INV-2024-0601
Vendor: CleanTech Solutions
Date: 2024-09-15

Items:
  1. Air Filter AF-200 -- Qty: 4 -- Unit: $35.00 -- Total: $140.00

Subtotal: $140.00
Tax (7%): $9.80
Total: $149.80

Payment: Paid (Check #4455, 2024-09-20)
""",
        "expected_behavior": "Should pass on first try. No retries needed.",
    },

    "doc_bad_date": {
        "text": """
INVOICE #INV-2024-0715
Vendor: FastParts Express
Date: July 15, 2024

Items:
  1. Bearing Set BS-100 -- Qty: 8 -- Unit: $15.00 -- Total: $120.00

Subtotal: $120.00
Tax (8%): $9.60
Total: $129.60

Payment: Unpaid -- Due August 15, 2024
""",
        "expected_behavior": "Date format error -> retry with feedback -> should fix.",
    },

    "doc_bad_math": {
        "text": """
INVOICE #INV-2024-0820
Vendor: BuildRight Hardware
Date: 2024-08-20

Items:
  1. Drill Bit Set -- Qty: 3 -- Unit: $25.00 -- Total: $75.00
  2. Safety Goggles -- Qty: 10 -- Unit: $12.00 -- Total: $120.00

Subtotal: $195.00
Tax (6%): $11.70
Shipping: $8.00
Total: $214.70

Payment: Paid
""",
        "expected_behavior": "Check: 195 + 11.70 + 8.00 = 214.70. This is correct. Should pass.",
    },

    "doc_no_details": {
        "text": """
Receipt from Mike's Auto Shop
Date: 2024-10-05
Oil change: $45.00
Paid cash
""",
        "expected_behavior": "Minimal doc. Many fields will be null. Should NOT retry for absent data.",
    },

    "doc_multi_error": {
        "text": """
INVOICE #INV-2024-0930
Vendor: Global Electronics Ltd.
Date: September 30, 2024

Items:
  1. USB-C Cable (2m) -- Qty: 100 -- Unit: $3.50 -- Total: $350.00
  2. HDMI Adapter -- Qty: 50 -- Unit: $8.00 -- Total: $400.00
  3. Wireless Mouse -- Qty: 25 -- Unit: $15.00 -- Total: $375.00

Subtotal: $1,125.00
Tax (8.5%): $95.63
Shipping: $20.00
Total: $1,240.63

Payment: Partial -- $500 deposit received Sept 15. Balance due Oct 30.
""",
        "expected_behavior": "Date format error + possibly more. Should retry and fix progressively.",
    },
}

print("Challenge documents loaded:")
for name, doc in CHALLENGE_DOCS.items():
    print(f"  {name}: {doc['expected_behavior']}")

In [ ]:
# ============================================================
# YOUR TURN: Implement the validation-retry pipeline
# ============================================================
#
# You have all the building blocks from Phase 2:
#   - validate_extraction(extraction, schema)
#   - should_retry(errors, source_document)
#   - build_retry_prompt(document, failed_extraction, errors)
#
# Task: Process all 5 documents through the pipeline and track:
#   - Which documents passed on first try
#   - Which needed retries (and how many)
#   - Which had errors that should NOT be retried
#   - What error patterns were detected (for upstream feedback)

def process_documents(docs, tool, schema, max_retries=2):
    """
    Process a batch of documents through the validation-retry pipeline.
    
    Returns a dict of results with:
      - extraction: the final extraction dict
      - retries: number of retries used
      - first_try_pass: True if no retries needed
      - error_patterns: list of error types detected (e.g., 'date_format', 'math_error', 'absent_data')
    """
    results = {}
    
    for doc_name, doc_info in docs.items():
        # TODO: Implement the processing loop for each document
        # 1. Extract using tool_use
        # 2. Validate
        # 3. Decide whether to retry
        # 4. If retrying, build retry prompt with error feedback
        # 5. Track error patterns
        pass  # Replace with your implementation
    
    return results


# Uncomment when ready:
# results = process_documents(CHALLENGE_DOCS, extraction_tool, EXTRACTION_SCHEMA)
# 
# print("=" * 60)
# print("PIPELINE RESULTS")
# print("=" * 60)
# for name, r in results.items():
#     print(f"\n{name}:")
#     print(f"  First try pass: {r.get('first_try_pass')}")
#     print(f"  Retries used: {r.get('retries')}")
#     print(f"  Error patterns: {r.get('error_patterns', [])}")
#     remaining = r.get('extraction', {}).get('_validation_errors')
#     if remaining:
#         print(f"  Remaining errors: {remaining}")

In [ ]:
# ============================================================
# CHECKER: Validate your pipeline implementation
# ============================================================

def check_pipeline(results):
    """Validate the pipeline processed documents correctly."""
    checks = []
    
    # Check 1: doc_perfect should pass on first try
    if "doc_perfect" in results:
        r = results["doc_perfect"]
        passed = r.get("first_try_pass", False) or r.get("retries", 99) == 0
        checks.append(("doc_perfect passes on first try", passed))
    else:
        checks.append(("doc_perfect is processed", False))
    
    # Check 2: doc_bad_date should have been retried
    if "doc_bad_date" in results:
        r = results["doc_bad_date"]
        retried = r.get("retries", 0) > 0
        has_date_pattern = "date_format" in r.get("error_patterns", [])
        checks.append(("doc_bad_date was retried for date format", retried))
        checks.append(("doc_bad_date detected 'date_format' pattern", has_date_pattern))
    else:
        checks.append(("doc_bad_date is processed", False))
    
    # Check 3: doc_no_details should NOT waste retries on absent data
    if "doc_no_details" in results:
        r = results["doc_no_details"]
        extraction = r.get("extraction", {})
        # Key check: warranty_expiry should be null, not fabricated
        warranty = extraction.get("warranty_expiry")
        no_fabrication = warranty is None
        checks.append(("doc_no_details: warranty_expiry is null (not fabricated)", no_fabrication))
    else:
        checks.append(("doc_no_details is processed", False))
    
    # Check 4: All 5 documents processed
    all_processed = len(results) == 5
    checks.append(("All 5 documents processed", all_processed))
    
    # Check 5: error_patterns populated
    any_patterns = any(r.get("error_patterns") for r in results.values())
    checks.append(("Error patterns tracked for feedback loop", any_patterns))
    
    # Report
    print("=" * 50)
    print("PIPELINE CHECK RESULTS")
    print("=" * 50)
    all_passed = True
    for desc, passed in checks:
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_passed = False
        print(f"  {status}: {desc}")
    
    print()
    if all_passed:
        print("ALL CHECKS PASSED")
    else:
        print("SOME CHECKS FAILED -- review your implementation.")
    return all_passed


# Uncomment when ready:
# check_pipeline(results)

---
## Phase 4: STRESS TEST

Edge cases that test the boundaries of your validation-retry logic.

In [ ]:
STRESS_DOCS = {
    "all_valid": {
        "text": """
INVOICE #INV-2024-1100
Vendor: Reliable Supply Co.
Date: 2024-11-01

Items:
  1. Printer Paper -- Qty: 20 -- Unit: $5.00 -- Total: $100.00

Subtotal: $100.00
Tax (6%): $6.00
Total: $106.00

Payment: Paid (2024-11-01)
Warranty: 1-year satisfaction guarantee. Expiry: 2025-11-01
""",
        "expect": "Pass on first try. All fields present and correct.",
    },

    "three_errors": {
        "text": """
INVOICE #INV-2024-1205
Vendor: MultiError Test Corp.
Date: December 5, 2024

Items:
  1. Widget A -- Qty: 10 -- Unit: $20.00 -- Total: $200.00
  2. Widget B -- Qty: 5 -- Unit: $30.00 -- Total: $150.00

Subtotal: $350.00
Tax (10%): $35.00
Total: $385.00

Payment: Paid December 10, 2024
""",
        "expect": "Date format errors (purchase_date + payment_date). Should fix with retry.",
    },

    "retry_makes_worse": {
        "text": """
Quick note: Bob owes us $200 for the repair job on Oct 3rd.
""",
        "expect": "Minimal document. Most fields absent. Should NOT retry -- retry would fabricate.",
    },
}

print("=" * 60)
print("STRESS TEST")
print("=" * 60)

for name, doc in STRESS_DOCS.items():
    print(f"\n--- {name} ---")
    print(f"Expected: {doc['expect']}")
    result = extract_with_retry(doc["text"], extraction_tool, max_retries=2)
    print(f"  Retries used: {result.get('_retries_used')}")
    remaining = result.get('_validation_errors')
    if remaining:
        print(f"  Remaining errors: {remaining}")
    else:
        print(f"  Status: clean extraction")
    print(f"  warranty_expiry: {result.get('warranty_expiry')!r}")

In [ ]:
# Stress test validation

print("=" * 60)
print("STRESS TEST VALIDATION")
print("=" * 60)

all_passed = True

# Test 1: all_valid should pass on first try
result_valid = extract_with_retry(STRESS_DOCS["all_valid"]["text"], extraction_tool, verbose=False)
if result_valid.get("_retries_used", 99) == 0 and "_validation_errors" not in result_valid:
    print("PASS: 'all_valid' passed on first try")
else:
    print(f"FAIL: 'all_valid' should pass first try. Retries={result_valid.get('_retries_used')}, errors={result_valid.get('_validation_errors')}")
    all_passed = False

# Test 2: three_errors should be fixed with retries
result_errors = extract_with_retry(STRESS_DOCS["three_errors"]["text"], extraction_tool, verbose=False)
purchase_date = result_errors.get("purchase_date", "")
try:
    datetime.strptime(purchase_date, "%Y-%m-%d")
    print(f"PASS: 'three_errors' purchase_date fixed to '{purchase_date}'")
except ValueError:
    print(f"FAIL: 'three_errors' purchase_date still wrong: '{purchase_date}'")
    all_passed = False

# Test 3: retry_makes_worse should NOT fabricate warranty
result_minimal = extract_with_retry(STRESS_DOCS["retry_makes_worse"]["text"], extraction_tool, verbose=False)
warranty = result_minimal.get("warranty_expiry")
if warranty is None:
    print("PASS: 'retry_makes_worse' warranty_expiry is null (no fabrication)")
else:
    print(f"FAIL: 'retry_makes_worse' warranty_expiry should be null, got {warranty!r}")
    all_passed = False

# Test 4: cost check -- minimal doc should use few retries
retries_minimal = result_minimal.get("_retries_used", 99)
if retries_minimal <= 1:
    print(f"PASS: 'retry_makes_worse' used {retries_minimal} retries (not wasting calls)")
else:
    print(f"WARN: 'retry_makes_worse' used {retries_minimal} retries (could save cost)")

print()
if all_passed:
    print("ALL STRESS TESTS PASSED")
else:
    print("SOME TESTS FAILED -- review output above.")

---
## Key Takeaways

1. **Blind retry does not work.** Without specific error feedback, the model makes the same mistakes repeatedly. Each retry costs money with no improvement.

2. **Never retry for absent information.** If the document does not contain warranty data, asking the model to "try harder" pushes it to fabricate. The should_retry function must distinguish fixable errors from absent data.

3. **The retry prompt must include:**
   - The original document (context)
   - The failed extraction (so the model sees what it produced)
   - Each specific validation error (so the model knows what to fix)
   - A correction instruction (not a "try again" instruction)

4. **Validation types and retry decisions:**
   - Date format wrong -> retry (fixable)
   - Math/total mismatch -> retry (fixable)
   - Invalid enum value -> retry (fixable)
   - Field absent from source -> do NOT retry (accept null)

5. **Track error patterns for upstream feedback.** If 40% of documents have date format errors, the problem may be in your schema description, not in the documents. Feed this signal back to improve the extraction prompt.

6. **On the exam:** When asked about retry strategies, the correct answer always includes specific error feedback in the retry prompt and a decision gate that prevents retrying for absent data.

## Next

Lab 1.5: Batch Processing -- scaling extraction across thousands of documents.